# Materials 01 — A perfect crystal and its cohesive energy

This series follows the arc of the LiveCoMS
[materials-science tutorials for LAMMPS](https://doi.org/10.33011/livecoms.6.1.3037)
(Gravelle, Tschopp & Kohlmeyer): **from a perfect crystal to failure** —
first a defect-free single crystal and its cohesive properties, then parameter
scans, the mechanical response of a strained crystal, grain boundaries and
their fracture, and finally the response of a surface to contact loading.

The original tutorials use an embedded-atom (EAM) potential for aluminum.
The wasm build shipped here doesn't include EAM, so we study a **model
Lennard-Jones metal** instead: an fcc crystal bound by the 12-6 LJ potential,
in LAMMPS *reduced units* (`units lj` — lengths in σ, energies in ε,
`mass = 1`). Every technique — and every line of LAMMPS input — carries over
unchanged to a real EAM metal; only the numbers wear different units. A
recurring theme of the original series holds here too: **results are only as
good as the interatomic potential.**

Start the engine:

In [ ]:
%pip install lammps-js

## Build an fcc crystal

A 4×4×4 supercell of fcc lattice sites (4 atoms per conventional cell → 256
atoms), with a first guess at the density. `run 0` evaluates energy and
pressure without moving anything:

In [ ]:
from lammps import lammps

lmp = await lammps(output=None)
lmp.commands_string("""
units         lj
atom_style    atomic
boundary      p p p
lattice       fcc 1.0          # in units lj this sets the reduced density
region        box block 0 4 0 4 0 4
create_box    1 box
create_atoms  1 box
mass          1 1.0
pair_style    lj/cut 2.5
pair_coeff    1 1 1.0 1.0 2.5
run           0 post no
""")

# In `units lj` LAMMPS normalizes extensive thermo output per atom.
print("atoms:                 ", lmp.get_natoms())
print("energy per atom (eps): ", round(lmp.get_thermo("pe"), 4))
print("pressure:              ", round(lmp.get_thermo("press"), 3))

The energy is negative (the crystal is bound) but the **pressure is not
zero**: our guessed density is not the equilibrium density for this
potential. The crystal "pushes" on the box. This mirrors the original
Tutorial 1, where guessing the experimental lattice constant of aluminum left
a 23 600 bar pressure.

## Relax the box: equilibrium lattice constant and cohesive energy

`fix box/relax` lets the minimizer adjust the box until the pressure
vanishes, while conjugate-gradient minimization keeps atoms at their lattice
sites:

In [ ]:
lmp.commands_string("""
fix relax all box/relax iso 0.0 vmax 0.001
min_style cg
minimize 1.0e-12 1.0e-12 10000 100000
""")

a0 = lmp.get_thermo("lx") / 4        # 4 conventional cells along x
ecoh = lmp.get_thermo("pe")          # per atom already (units lj)
print("equilibrium lattice constant a0:", round(a0, 5), "sigma")
print("cohesive energy E_coh:          ", round(ecoh, 5), "eps/atom")
print("residual pressure:              ", f'{lmp.get_thermo("press"):.2e}')

Two numbers characterize the crystal: **a₀ ≈ 1.5496 σ** (nearest-neighbor
distance a₀/√2 ≈ 1.096 σ, close to the LJ dimer minimum 2^{1/6} ≈ 1.122 σ —
squeezed slightly by the attraction of further shells) and
**E_coh ≈ −8.10 ε/atom** (the full-lattice sum gives −8.61 ε; our cutoff of
2.5 σ discards the tail).

## The potential *is* the model

The original tutorial's key lesson: repeat the calculation with three
different aluminum potentials and you get three different answers. Our
version of that experiment — vary the cutoff radius:

In [ ]:
results = {}
for rc in (2.0, 2.5, 3.0, 4.0):
    lmp2 = await lammps(output=None)
    lmp2.commands_string(f"""
units lj
atom_style atomic
lattice fcc 1.0
region box block 0 4 0 4 0 4
create_box 1 box
create_atoms 1 box
mass 1 1.0
pair_style lj/cut {rc}
pair_coeff 1 1 1.0 1.0 {rc}
fix relax all box/relax iso 0.0 vmax 0.001
min_style cg
minimize 1.0e-12 1.0e-12 10000 100000
""")
    results[rc] = (lmp2.get_thermo("lx") / 4, lmp2.get_thermo("pe"))
    lmp2.close()

print(f'{"cutoff":>8} {"a0":>10} {"E_coh":>10}')
for rc, (a, e) in results.items():
    print(f"{rc:>8} {a:>10.5f} {e:>10.5f}")

Same crystal, same functional form — a percent-level spread in a₀ and a
~10 % spread in cohesive energy, purely from the cutoff. *Always check that
a potential reproduces the properties you care about.*

Keep the values from the 2.5 σ run in mind — the rest of the series uses
them:

- **a₀ = 1.5496 σ**, **E_coh = −8.0998 ε/atom**

Next: [02 — The energy–volume curve](02-energy-volume-curve.ipynb), where a
Python loop replaces LAMMPS `jump` loops for parameter scans.

In [ ]:
lmp.close()